In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# ClinicalDistill — LLaMA-3.2-1B LoRA
## Experiment 7

Third model family in the benchmark — Meta LLaMA alongside Google Gemma and Alibaba Qwen.

| Model             | Lab     | Params | Valid JSON | F1    | Urgent Acc |
|-------------------|---------|--------|------------|-------|------------|
| Gemma-3-1B LoRA   | Google  | 1B     | 100%       | 0.781 | 85.7%      |
| Gemma-3-1B QLoRA  | Google  | 1B     | 100%       | 0.740 | 82.9%      |
| Qwen1.5-1.8B LoRA | Alibaba | 1.8B   | 100%       | 0.707 | 74.3%      |
| Qwen1.5-1.8B QLoRA| Alibaba | 1.8B   | 94.3%      | 0.696 | 87.9%      |
| LLaMA-3.2-1B LoRA | Meta    | 1B     | ___        | ___   | ___        |

**Before running:** Accept Meta's license at huggingface.co/meta-llama/Llama-3.2-1B-Instruct (takes 30 sec, usually instant approval). Add HF_TOKEN to Colab Secrets.

In [3]:
# Cell 1 — Install (no os.kill at the end)
!pip install -q transformers peft accelerate bitsandbytes datasets trl huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.5 MB/s eta 0:00:00


In [ ]:
# Install then restart — Colab's cached packages need replacing
!pip install -q --force-reinstall bitsandbytes
!pip install -q transformers peft accelerate datasets trl huggingface_hub

import os
os.kill(os.getpid(), 9)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/

In [4]:
# Run from here after runtime restarts
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill'

print(os.listdir(PROJECT_PATH))
!wc -l {PROJECT_PATH}/data/train_fixed.jsonl {PROJECT_PATH}/data/test_fixed.jsonl

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['README.md', 'requirements.txt', '.env', '.gitignore', 'data', 'schema', '.git', 'models']
  145 /content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill/data/train_fixed.jsonl
   35 /content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill/data/test_fixed.jsonl
  180 total


In [5]:
import torch

print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Tesla T4
VRAM: 15.6 GB


In [6]:
from datasets import load_dataset
import json

dataset = load_dataset("json", data_files={
    "train": f"{PROJECT_PATH}/data/train_fixed.jsonl",
    "test":  f"{PROJECT_PATH}/data/test_fixed.jsonl"
})

print(dataset)
print(dataset["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'metadata'],
        num_rows: 145
    })
    test: Dataset({
        features: ['input', 'output', 'metadata'],
        num_rows: 35
    })
})
{'input': 'Patient presents with chest pain radiating to the left arm and shortness of breath for the past 2 hours.', 'output': {'symptoms': ['chest pain', 'shortness of breath'], 'duration': ['2 hours', 'unspecified'], 'severity': ['severe', 'unspecified'], 'urgent': True}, 'metadata': {'clinical_domain': 'cardiac', 'generated_by': 'gpt-4o'}}


In [7]:
from huggingface_hub import login
import json
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
def format_example(example):
    output = example['output']
    if isinstance(output, str):
        output = json.loads(output)
    return {
        "text": f"""<instruction>
Extract symptoms from the clinical note below. Reply with ONLY valid JSON. No comments, no markdown, no extra text.
Format: {{"symptoms": ["s1", "s2"], "duration": ["d1", "d2"], "severity": ["sev1", "sev2"], "urgent": true/false}}
Use "unspecified" if duration or severity unknown. urgent=true only for chest pain, breathing difficulty, stroke, severe bleeding.
</instruction>
<input>{example['input']}</input>
<output>{json.dumps(output)}</output>"""
    }

dataset = dataset.map(format_example)
print(dataset["train"][0]["text"])

Map:   0%|          | 0/145 [00:00<?, ? examples/s]

Map:   0%|          | 0/35 [00:00<?, ? examples/s]

<instruction>
Extract symptoms from the clinical note below. Reply with ONLY valid JSON. No comments, no markdown, no extra text.
Format: {"symptoms": ["s1", "s2"], "duration": ["d1", "d2"], "severity": ["sev1", "sev2"], "urgent": true/false}
Use "unspecified" if duration or severity unknown. urgent=true only for chest pain, breathing difficulty, stroke, severe bleeding.
</instruction>
<input>Patient presents with chest pain radiating to the left arm and shortness of breath for the past 2 hours.</input>
<output>{"symptoms": ["chest pain", "shortness of breath"], "duration": ["2 hours", "unspecified"], "severity": ["severe", "unspecified"], "urgent": true}</output>


In [10]:
from huggingface_hub import model_info

try:
    info = model_info("meta-llama/Llama-3.2-1B-Instruct")
    print("Access confirmed ✅")
    print(f"Model: {info.modelId}")
except Exception as e:
    print(f"No access yet ❌: {e}")

Access confirmed ✅
Model: meta-llama/Llama-3.2-1B-Instruct


In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import userdata
import torch

model_id = "meta-llama/Llama-3.2-1B-Instruct"
hf_token = userdata.get('HF_TOKEN')  # ← get token explicitly

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=hf_token  # ← pass token here
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=hf_token  # ← pass token here too
)
model.config.pad_token_id = tokenizer.pad_token_id

print(f"Model: {model_id}")
print(f"Parameters: {model.num_parameters():,}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model: meta-llama/Llama-3.2-1B-Instruct
Parameters: 1,235,814,400
VRAM used: 2.47 GB


In [18]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,703,936 || all params: 1,237,518,336 || trainable%: 0.1377


In [19]:
from trl import SFTTrainer, SFTConfig
import time

OUTPUT_DIR = f"{PROJECT_PATH}/models/llama-lora"

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,    # LLaMA-3.2 uses bfloat16 — use bf16 not fp16
    fp16=False,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
)

start_time = time.time()
print("Starting LLaMA-3.2-1B LoRA training...")
trainer.train()
training_time = time.time() - start_time

print(f"Training complete: {training_time:.0f}s ({training_time/60:.1f} min)")
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Adding EOS to train dataset:   0%|          | 0/145 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/145 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Starting LLaMA-3.2-1B LoRA training...


Step,Training Loss
10,1.956485
20,0.992041
30,0.500820
40,0.346811
50,0.327915
60,0.292424
70,0.299304
80,0.303566
90,0.275477


Training complete: 469s (7.8 min)
VRAM: 2.51 GB


In [20]:
SAVE_PATH = f"{PROJECT_PATH}/models/llama-lora-final"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Saved to: {SAVE_PATH}")

Saved to: /content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill/models/llama-lora-final


In [21]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

SAVE_PATH = f"{PROJECT_PATH}/models/llama-lora-final"

tokenizer_loaded = AutoTokenizer.from_pretrained(SAVE_PATH)
tokenizer_loaded.pad_token = tokenizer_loaded.eos_token
tokenizer_loaded.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
base_model.config.pad_token_id = tokenizer_loaded.pad_token_id
model_loaded = PeftModel.from_pretrained(base_model, SAVE_PATH)
model_loaded.eval()

print("Model loaded for inference")
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Model loaded for inference
VRAM: 4.99 GB


In [22]:
def extract_clinical(text):
    prompt = f"""<instruction>
Extract symptoms from the clinical note below. Reply with ONLY valid JSON. No comments, no markdown, no extra text.
Format: {{"symptoms": ["s1", "s2"], "duration": ["d1", "d2"], "severity": ["sev1", "sev2"], "urgent": true/false}}
Use "unspecified" if duration or severity unknown. urgent=true only for chest pain, breathing difficulty, stroke, severe bleeding.
</instruction>
<input>{text}</input>
<output>"""

    inputs = tokenizer_loaded(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model_loaded.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer_loaded.eos_token_id
        )
    response = tokenizer_loaded.decode(outputs[0], skip_special_tokens=True)
    return response.split("<output>")[-1].strip()


test_cases = [
    "Elderly patient complaining of severe headache for 2 days, blurred vision, and confusion.",
    "Child presents with mild fever since yesterday, runny nose and occasional cough.",
    "Patient reports sharp lower back pain for a week, worse when sitting, no fever."
]
for case in test_cases:
    print(f"INPUT: {case}")
    print(f"OUTPUT: {extract_clinical(case)}")
    print("-" * 60)

INPUT: Elderly patient complaining of severe headache for 2 days, blurred vision, and confusion.
OUTPUT: {"symptoms": ["headache", "blurred vision", "confusion"], "duration": ["2 days", "unspecified", "unspecified"], "severity": ["severe", "unspecified", "unspecified"], "urgent": true}</output>
------------------------------------------------------------
INPUT: Child presents with mild fever since yesterday, runny nose and occasional cough.
OUTPUT: {"symptoms": ["fever", "runny nose", "cough"], "duration": ["since yesterday", "unspecified", "unspecified"], "severity": ["mild", "unspecified", "unspecified"], "urgent": false}</output>
------------------------------------------------------------
INPUT: Patient reports sharp lower back pain for a week, worse when sitting, no fever.
OUTPUT: {"symptoms": ["lower back pain"], "duration": ["a week"], "severity": ["unspecified"], "urgent": true}</output>
------------------------------------------------------------


In [23]:
import json

def parse_output(text):
    try:
        text = text.strip().replace("</output>", "").strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        return json.loads(text.strip()), True
    except:
        return {}, False

def flatten_symptoms(symptoms):
    flat = []
    for s in symptoms:
        if isinstance(s, str):
            flat.append(s.lower())
        elif isinstance(s, dict):
            name = s.get("symptom") or s.get("name") or s.get("description") or ""
            if name:
                flat.append(name.lower())
    return flat

def symptom_f1(pred_symptoms, true_symptoms):
    pred_set = set(flatten_symptoms(pred_symptoms))
    true_set = set(flatten_symptoms(true_symptoms))
    tp = len(pred_set & true_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall    = tp / (tp + fn) if tp + fn > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if precision + recall > 0 else 0
    return round(f1, 3)

valid_json_count = 0
f1_scores = []
urgent_correct = 0

for example in dataset["test"]:
    raw_output = extract_clinical(example["input"])
    pred, is_valid = parse_output(raw_output)
    true = example["output"]
    if is_valid:
        valid_json_count += 1
        f1_scores.append(symptom_f1(pred.get("symptoms", []), true.get("symptoms", [])))
        if pred.get("urgent") == true.get("urgent"):
            urgent_correct += 1

total = len(dataset["test"])
avg_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0

print("=" * 50)
print("EVALUATION RESULTS — LLaMA-3.2-1B LoRA")
print("=" * 50)
print(f"Valid JSON rate : {valid_json_count}/{total} ({100*valid_json_count/total:.1f}%)")
print(f"Avg Symptom F1  : {avg_f1:.3f}")
print(f"Urgent Accuracy : {urgent_correct}/{valid_json_count} ({100*urgent_correct/max(valid_json_count,1):.1f}%)")
print("=" * 50)

EVALUATION RESULTS — LLaMA-3.2-1B LoRA
Valid JSON rate : 35/35 (100.0%)
Avg Symptom F1  : 0.743
Urgent Accuracy : 26/35 (74.3%)
